# 01 — Generate Synthetic Insurance Claim Payloads

Produces 50 synthetic claim JSON blobs shaped like `SamplePayload.json` (Guidewire
ClaimCenter homeowners claim), but with template-generated realistic insurance text
instead of Lorem ipsum. Used as input fixtures for the `claims_fe` feature-engineering
pyfunc endpoint.

**Marker injection**: each payload gets 0–3 random "markers" (attorney LOR phrases,
SIU red flags, medical escalation, subrogation language, urgency language, dollar
amounts, health disclosures). The markers_injected field lets notebook 04 assert that
the corresponding feature extractors fire.

**Outputs:**
- `/Volumes/fins_genai/classic_ml/fe_test_payloads/*.json` — individual claim JSONs
- `fins_genai.classic_ml.fe_test_payloads` — Delta table `(payload_id, lob_type, loss_cause, markers_injected, payload_json)`

In [0]:
CATALOG = "fins_genai"
SCHEMA = "classic_ml"
VOLUME_NAME = "fe_test_payloads"
TABLE_NAME = "fe_test_payloads"
N_PAYLOADS = 50
SEED = 42

# Text-size knobs. Each unit of "repeats" adds one random fragment to the blob.
# Rough sizing on the default templates:
#   n_notes=8,  n_docs=6   ->  ~245 words / ~1.9 KB combined text per payload  (quick tests)
#   n_notes=80, n_docs=60  ->  ~2.5K words / ~20 KB                            (realistic claim size)
#   n_notes=400, n_docs=300 -> ~13K words / ~100 KB                            (long-tail real claim)
#   n_notes=1600, n_docs=1200 -> ~53K words / ~400 KB                          (stress / latency test)
# Scaling is linear in repeats: the regex extractors cost ~0.5 ms per KB of text.
NOTE_REPEATS = 80
DOC_REPEATS = 60

VOLUME_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME_NAME}"

In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.{VOLUME_NAME}")

DataFrame[]

## Template banks

Per-LOB/cause phrase libraries. Kept small and obviously synthetic but lexically
realistic — the goal is to exercise the feature extractors, not to fool a human.

In [0]:
# LOB / loss-cause combinations we'll generate. Maps to realistic CLM_SBTYP_CD values.
LOB_COMBOS = [
    {
        "lob":        "HomeownersLine_HOE",
        "loss_cause": "Water_Ext",
        "sub_type":   "WaterAndFreezing_Ext",
        "group":      "Foremost",
        "incident_phrases": [
            "water heater burst in basement; water reached subfloor and carpets",
            "supply line under kitchen sink failed overnight; cabinets and flooring wet",
            "dishwasher supply hose ruptured; hardwood floor warped across kitchen",
            "toilet overflow on second floor; ceiling damage to first-floor living room",
            "washing machine drain backed up; laundry room flooring and drywall soaked",
        ],
        "doc_fragments": [
            "Water mitigation estimate from contractor: drying equipment placed for 4 days, antimicrobial applied to affected areas.",
            "Inspection report notes sub-floor saturation under kitchen cabinets; moisture readings exceed 25 percent in lower plates.",
            "Emergency plumber invoice covers after-hours service, replacement supply valve, and haul-away of damaged base cabinets.",
            "Moisture mapping survey shows damage extending 8 feet into adjacent hallway; recommend pulling baseboards and drying cavity.",
        ],
        "note_fragments": [
            "Insured contacted to schedule inspection; left voicemail with claim number and callback options.",
            "Spoke with insured's spouse; dryer equipment running in kitchen and hallway; insured requests contractor referral list.",
            "Reviewed water mitigation invoice; amount appears consistent with industry standards for Cat 2 water loss of this scope.",
            "Coverage review: dwelling and contents in play; ALE coverage triggered for five nights of alternate lodging.",
            "Follow-up with contractor on scope expansion for affected hallway and adjacent bathroom vanity; awaiting updated estimate.",
        ],
        "water_source": "other",
    },
    {
        "lob":        "HomeownersLine_HOE",
        "loss_cause": "Fire_Ext",
        "sub_type":   "Fire_Ext",
        "group":      "Foremost",
        "incident_phrases": [
            "kitchen fire started from unattended cooking on gas range; heavy smoke throughout first floor",
            "electrical fire in attached garage; fire department responded and contained damage to garage and shared wall",
            "chimney fire during cold snap; attic space affected with minor structural charring",
            "candle ignited curtains in living room; fire spread to adjacent wall before extinguishment",
        ],
        "doc_fragments": [
            "Fire department incident report notes point of origin as stovetop; cause listed as unattended cooking with grease ignition.",
            "Cleaning and restoration estimate covers smoke-damage treatment for textiles, ozone treatment for affected rooms, and HVAC duct cleaning.",
            "Structural engineer assessment indicates wall between garage and dwelling retained integrity; sheathing replacement recommended.",
        ],
        "note_fragments": [
            "Insured advised of ALE eligibility; hotel arrangements made while smoke mitigation proceeds.",
            "Contents inventory in progress; insured providing photographs and receipts for damaged electronics and textiles.",
            "Public adjuster contact attempted callback; confirmed carrier position on scope limits.",
            "Reviewed salvage list with restoration vendor; non-salvageable contents include kitchen linens, small appliances, and pantry goods.",
        ],
        "water_source": None,
    },
    {
        "lob":        "HomeownersLine_HOE",
        "loss_cause": "Wind_Ext",
        "sub_type":   "WindHail_Ext",
        "group":      "Foremost",
        "incident_phrases": [
            "severe thunderstorm with straight-line winds removed roofing shingles across south elevation",
            "large limb from neighbor tree fell on dwelling during storm, puncturing roof decking",
            "hail storm caused dimpling of gutters, downspouts, and air conditioning condenser fins",
            "tornado warning event; fence and outbuildings destroyed, minor damage to main dwelling roof",
        ],
        "doc_fragments": [
            "Roofer estimate lists 22 squares of architectural shingle replacement plus ice-and-water shield on eaves.",
            "HVAC inspection notes condenser coil fin damage consistent with hail; recommend replacement over repair.",
            "Arborist report on fallen limb origin and tree health; attached client consent for debris removal.",
            "Adjuster roof inspection photos uploaded; documented hail bruising pattern on north and west slopes.",
        ],
        "note_fragments": [
            "Insured requested tarp service for exposed sheathing pending roofer mobilization; vendor dispatched same day.",
            "Comparative review of contractor bids; wide range due to supplement for decking replacement.",
            "Received contractor supplement for decking beneath damaged shingles; within reserve.",
            "Discussed depreciation and recoverable depreciation schedule with insured; provided written summary.",
        ],
        "water_source": None,
    },
    {
        "lob":        "HomeownersLine_HOE",
        "loss_cause": "Theft_Ext",
        "sub_type":   "TheftFromDwelling_Ext",
        "group":      "Foremost",
        "incident_phrases": [
            "dwelling burglarized while insureds traveled for the weekend; point of entry via rear sliding door",
            "break-in through basement window; items taken include electronics, jewelry, and cash kept in home office",
            "unforced entry through unlocked garage side door during afternoon hours; stolen items from interior",
            "overnight theft; multiple tools taken from detached workshop, plus small safe from master bedroom",
        ],
        "doc_fragments": [
            "Police report number attached; investigating officer assigned, evidence technicians collected prints.",
            "Inventory of stolen items with purchase dates, serial numbers where available, and replacement-cost research.",
            "Security system report shows alarm not armed at time of event; motion detector activity log included.",
            "Appraisal documents for jewelry items provided by insured; dates and valuations pre-loss.",
        ],
        "note_fragments": [
            "Contacted insured to walk through required proof-of-loss documentation and sworn statement timeline.",
            "Reviewed inventory against coverage sublimits; jewelry schedule in effect, cash sublimit applies.",
            "Insured provided updated list with serial numbers for two stolen laptops; running serial-match check.",
            "Follow-up call scheduled to review sworn statement and proof-of-loss prior to claim closure window.",
        ],
        "water_source": None,
    },
    {
        "lob":        "HomeownersLine_HOE",
        "loss_cause": "Liability_Ext",
        "sub_type":   "PremisesLiability_Ext",
        "group":      "Foremost",
        "incident_phrases": [
            "visitor slipped on wet entryway tile and reported back pain; ambulance called to scene",
            "neighbor child fell from rear deck; report of wrist injury taken at urgent care",
            "dog off-leash interaction with mail carrier; minor hand contact reported, no bite confirmed",
            "guest tripped on loose stair tread; reported head strike with brief loss of consciousness",
        ],
        "doc_fragments": [
            "Urgent care intake notes documenting claimant complaints of lower back stiffness and referred pain to hip.",
            "Incident witness statements from two household members corroborate sequence of events at entryway.",
            "Photographs of tile condition post-incident; cleaning schedule and signage documentation provided by insured.",
            "Medical bill statement listing initial urgent care visit, X-rays, and referral to physical therapy provider.",
        ],
        "note_fragments": [
            "Claimant contacted directly; walk-through of medical treatment plan and bill-forwarding process.",
            "Insured declined direct negotiation with claimant and referred all contact to carrier per policy terms.",
            "Reserved for potential medical payments and bodily injury liability; reassessment upon receipt of records.",
            "Reviewed claimant's prior medical history disclosure; will request full records via signed authorization.",
        ],
        "water_source": None,
    },
]

## Marker fragments

Each marker type injects a short phrase into the notes blob. Notebook 04 uses the
`markers_injected` column to verify the corresponding feature flag lit up.

In [0]:
MARKER_FRAGMENTS = {
    "attorney": [
        "Claimant advised that they have retained counsel; all further communication should be directed to our client's attorney.",
        "Received letter of representation (LOR) from law firm representing claimant; instructed to cease direct contact per their demand letter.",
        "Spouse contacted office to advise attorney representing family will be handling communications going forward.",
    ],
    "siu": [
        "Flagged for review: insured unable to produce receipts for high-value items; no police report filed at time of initial incident.",
        "Noted: recent policy inception, cash-only transactions referenced by insured, and prior claim history on adjacent property under investigation.",
        "Witness referenced in first report is unable to be located for follow-up; staged-loss indicators under review by SIU.",
    ],
    "medical": [
        "Claimant reports ER visit followed by MRI; orthopedic consult scheduled and physical therapy initiated three times per week.",
        "Treatment escalated to surgery consultation; inpatient hospitalization anticipated pending specialist review.",
        "Chiropractor and physical therapy ongoing; claimant reports persistent symptoms consistent with permanent injury designation.",
    ],
    "subrogation": [
        "Third party identified as likely at fault; contractor installation appears defective and manufacturer notification pending.",
        "Other driver's carrier contacted for subrogation; negligent operation confirmed by responding officer at scene.",
        "Product liability angle under review; defective component from manufacturer may trigger subrogation recovery.",
    ],
    "urgency": [
        "URGENT: claimant requests callback TODAY regarding temporary lodging; family includes elderly parent requiring accessible accommodations!!",
        "Time-sensitive ASAP: displacement ongoing, requesting immediate ALE advance and contractor dispatch right away!",
        "URGENT urgency on mold mitigation - cannot delay due to health risk; please expedite contractor approval immediately!!",
    ],
    "dollar": [
        "Contractor estimate totals $12,450.00 for remediation; supplement request of $3,875.25 for additional scope anticipated.",
        "Initial reserve set at $45,000; contents inventory pending may increase exposure by approximately $8,200 based on preliminary list.",
        "Emergency mitigation invoice of $2,340.50 paid; dwelling estimate pending review, currently estimated near $18,700 for full scope.",
    ],
    "health": [
        "Claimant discloses cancer treatment with chemotherapy; immunocompromised status noted, requesting expedited relocation away from affected area.",
        "Insured is pregnant and reports ongoing medication regimen; health considerations noted for temporary housing placement.",
        "Household member has disability and uses mobility assistance; accessible lodging required for ALE arrangement.",
    ],
}

ALL_MARKER_TYPES = list(MARKER_FRAGMENTS.keys())

## Synthetic payload builder

In [0]:
import json
import os
import random
from datetime import datetime, timedelta

random.seed(SEED)


def build_exposures(sub_type: str, claim_nk: str, claim_id: int) -> list:
    # Theft: contents + dwelling only. Liability: bodily-injury + med-pay only.
    # Everything else (water/fire/wind): dwelling + contents + living-expenses.
    if "Theft" in sub_type:
        types = ["Content", "Dwelling"]
    elif "Liability" in sub_type:
        types = ["BodilyInjury_Ext", "MedicalPayments_Ext"]
    else:
        types = ["Dwelling", "Content", "LivingExpenses"]
    exposures = []
    for i, t in enumerate(types):
        exposures.append({
            "_ID":             claim_id * 10 + i,
            "NK_CLM_EXPSR_ID": f"{claim_nk}-exp{i+1}",
            "NK_CLM_ID":       claim_nk,
            "EXPSR_TYP_CD":    t,
        })
    return exposures


def build_text_blob(fragments: list, n_repeats: int, extra_lines: list = None) -> str:
    """Assemble a realistic-looking note/document blob."""
    lines = []
    for _ in range(n_repeats):
        lines.append(random.choice(fragments))
    if extra_lines:
        # Interleave extra (marker) lines into the body
        for ln in extra_lines:
            insert_at = random.randint(0, max(len(lines) - 1, 0))
            lines.insert(insert_at, ln)
    return " ".join(lines)


def build_payload(idx: int) -> dict:
    combo = random.choice(LOB_COMBOS)

    claim_id = 10_000_000 + idx
    nk = f"ccp:{claim_id}"
    claim_num = f"{claim_id}-1"

    # Loss date in last 180 days; reported 0-21 days later
    loss_date = datetime(2025, 1, 1) + timedelta(days=random.randint(0, 180))
    report_delay_days = random.choices([0, 1, 3, 5, 10, 21], weights=[3, 4, 3, 2, 1, 1])[0]
    reported_date = loss_date + timedelta(days=report_delay_days, hours=random.randint(0, 23), minutes=random.randint(0, 59))

    # Construction year — random distribution skewed toward older homes for variety
    construction_year = random.choice([1955, 1963, 1978, 1984, 1995, 2001, 2008, 2015, 2019])

    # Pick 0-3 markers per payload
    n_markers = random.choices([0, 1, 2, 3], weights=[1, 3, 3, 2])[0]
    markers = random.sample(ALL_MARKER_TYPES, n_markers)
    marker_lines = [random.choice(MARKER_FRAGMENTS[m]) for m in markers]

    incident = random.choice(combo["incident_phrases"])
    incident_full = f"{incident}. Additional detail: affected areas include subfloor, baseboards, walls, and adjacent finishes."

    # Note blob: larger, with marker injection
    notes = build_text_blob(combo["note_fragments"], n_repeats=NOTE_REPEATS, extra_lines=marker_lines)

    # Document blob: no markers (markers only go in notes)
    docs = build_text_blob(combo["doc_fragments"], n_repeats=DOC_REPEATS)

    state = random.choice(["CA", "TX", "FL", "NY", "WA", "IL", "OH", "GA", "PA", "AZ"])

    claim = {
        "ID": claim_id,
        "NK_CLM_ID": nk,
        "CLM_NUM": claim_num,
        "CLM_DESC_TXT": incident[:120],
        "CLM_LOB_TYP_CD": combo["lob"],
        "CLM_LOSS_DTTM": loss_date.strftime("%Y-%m-%dT%H:%M:%S"),
        "CLM_RPRTD_DTTM": reported_date.strftime("%Y-%m-%dT%H:%M:%S.%f0"),
        "LOSS_CSE_TYP_CD": combo["loss_cause"],
        "CLM_SBTYP_CD": combo["sub_type"],
        "CLM_GRP_TYP_CD": combo["group"],
        "PLCY_STATE": state,
        "JURISDCTN_STATE_TYP_CD": state,
        "PRPTY_CNSTRCTN_YR": construction_year,
        "WATER_SRC_TYP_CD": combo["water_source"],
        "ConcatenatedDocs": docs,
        "ConcatenatedIncidents": incident_full,
        "ConcatenatedNotes": notes,
        "Exposures": build_exposures(combo["sub_type"], nk, claim_id),
    }

    return {
        "payload_id": f"SYN-{idx:03d}",
        "lob_type": combo["lob"],
        "loss_cause": combo["loss_cause"],
        "markers_injected": markers,
        "claim": claim,
    }


## Generate all payloads

In [0]:
payloads = [build_payload(i) for i in range(N_PAYLOADS)]

# Distribution sanity-check
from collections import Counter
print("LOB/cause distribution:")
for (lob, cause), cnt in Counter((p["lob_type"], p["loss_cause"]) for p in payloads).most_common():
    print(f"  {lob} / {cause}: {cnt}")

print("\nMarker-injection distribution:")
marker_counts = Counter()
for p in payloads:
    for m in p["markers_injected"]:
        marker_counts[m] += 1
    if not p["markers_injected"]:
        marker_counts["(none)"] += 1
for m, cnt in marker_counts.most_common():
    print(f"  {m}: {cnt}")

LOB/cause distribution:
  HomeownersLine_HOE / Wind_Ext: 13
  HomeownersLine_HOE / Fire_Ext: 10
  HomeownersLine_HOE / Water_Ext: 10
  HomeownersLine_HOE / Liability_Ext: 10
  HomeownersLine_HOE / Theft_Ext: 7

Marker-injection distribution:
  health: 14
  medical: 14
  attorney: 12
  dollar: 12
  urgency: 12
  subrogation: 10
  (none): 8
  siu: 6


## Write JSON files to UC Volume

In [0]:
os.makedirs(VOLUME_PATH, exist_ok=True)

for p in payloads:
    path = f"{VOLUME_PATH}/{p['payload_id']}.json"
    with open(path, "w") as f:
        # Same single-claim-wrapper shape as SamplePayload.json
        json.dump({"Claims": [p["claim"]]}, f)

written = sorted(os.listdir(VOLUME_PATH))
print(f"Wrote {len(written)} files; sample: {written[:3]}")

Wrote 50 files; sample: ['SYN-000.json', 'SYN-001.json', 'SYN-002.json']


## Write Delta table (one row per payload)

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, ArrayType

schema = StructType([
    StructField("payload_id", StringType(), False),
    StructField("lob_type", StringType(), True),
    StructField("loss_cause", StringType(), True),
    StructField("markers_injected", ArrayType(StringType()), True),
    StructField("payload_json", StringType(), False),
])

rows = [
    (
        p["payload_id"],
        p["lob_type"],
        p["loss_cause"],
        p["markers_injected"],
        json.dumps({"Claims": [p["claim"]]}),
    )
    for p in payloads
]
df = spark.createDataFrame(rows, schema=schema)
df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    f"{CATALOG}.{SCHEMA}.{TABLE_NAME}"
)
print(f"Wrote {CATALOG}.{SCHEMA}.{TABLE_NAME}: {df.count()} rows")

Wrote fins_genai.classic_ml.fe_test_payloads: 50 rows


In [0]:
display(spark.table(f"{CATALOG}.{SCHEMA}.{TABLE_NAME}").select("payload_id", "lob_type", "loss_cause", "markers_injected").limit(10))

payload_id,lob_type,loss_cause,markers_injected
SYN-000,HomeownersLine_HOE,Wind_Ext,List()
SYN-001,HomeownersLine_HOE,Fire_Ext,"List(attorney, health)"
SYN-002,HomeownersLine_HOE,Fire_Ext,List(dollar)
SYN-003,HomeownersLine_HOE,Water_Ext,List()
SYN-004,HomeownersLine_HOE,Wind_Ext,"List(siu, medical, attorney)"
SYN-005,HomeownersLine_HOE,Water_Ext,List()
SYN-006,HomeownersLine_HOE,Water_Ext,List(dollar)
SYN-007,HomeownersLine_HOE,Liability_Ext,"List(urgency, health)"
SYN-008,HomeownersLine_HOE,Liability_Ext,"List(medical, urgency, health)"
SYN-009,HomeownersLine_HOE,Fire_Ext,"List(health, medical)"
